# Quantum Graph Neural Networks (QGNNs) for Time-Series Anomaly Detection

In systems like smart power grids, telecommunication networks, or industrial IoT sensor arrays, anomalies rarely happen in isolation; they propagate along physical or structural lines. Classical GNNs capture these spatial dependencies by passing "messages" between nodes. A Quantum GNN advances this by maps sensor nodes to qubits, allowing quantum entanglement and interference to naturally model non-local, higher-order correlations between channels that classical networks struggle to track.

We will build a hybrid research pipeline:

Generate a Synthetic Multivariate Signal Dataset mapped to a structural graph, incorporating subtle localized anomalies.

Construct a Permutation-Invariant QGNN Ansatz using Amazon Braket and PennyLane.

Train the Network to predict future node states based on neighborhood topological structures, flagging deviations as anomalies.

## The Core Concept: Quantum Message Passing
In a QGNN, each node $i$ holds a state vector encoded into qubits. The quantum "message passing" step uses a multi-qubit entangling gate between connected nodes to simulate communication across edges.

### Step 1: Synthesize a Networked Signal Dataset

We will create a 4-node network (e.g., 4 synchronized sensors on an industrial asset). Nodes 0 and 1 are tightly connected, meaning their signals should correlate. An anomaly will be introduced where that structural correlation breaks.

In [ ]:
# Install required packages (quiet mode for cleaner output)
!pip install --quiet pennylane
!pip install --quiet amazon-braket-pennylane-plugin

In [ ]:
import pennylane as qml
from pennylane import numpy as np

# 1. Define the Graph Structure (Adjacency Matrix)
# Node 0 connects to 1 and 2. Node 1 connects to 0 and 3.
adjacency_matrix = np.array([
    [0, 1, 1, 0],
    [1, 0, 0, 1],
    [1, 0, 0, 0],
    [0, 1, 0, 0]
])
num_nodes = 4
time_steps = 100

# 2. Generate normal correlated time-series signals
np.random.seed(42)
time = np.linspace(0, 10, time_steps)
base_signal = np.sin(time)

data = np.zeros((time_steps, num_nodes))
for i in range(num_nodes):
    # Add minor phase shifts and normal distribution noise
    data[:, i] = base_signal + np.random.normal(0, 0.1, time_steps)

# 3. Inject an structural anomaly at time step 50
# Node 0 and Node 1 suddenly decouple completely
data[50:55, 0] += 2.5  # Spike anomaly on Node 0
data[50:55, 1] -= 2.5  # Inverse spike on Node 1

print(f"Dataset generated. Shape: {data.shape} (Time Steps, Nodes)")

### Step 2: The Quantum GNN Architecture
Our QGNN encodes the sensor readings at time $t$ as rotation angles, then applies a Quantum Graph Convolution Layer where entangling $CZ$ (Controlled-Z) gates are applied only if an edge exists between those nodes in our adjacency matrix.

In [ ]:
dev = qml.device("braket.local.qubit", wires=num_nodes)

@qml.qnode(dev)
def qgnn_layer(node_features, weights, adj):
    # Phase 1: Quantum Feature Mapping (Embedding node data)
    for i in range(num_nodes):
        qml.RX(node_features[i], wires=i)

    # Phase 2: Quantum Graph Convolution (Entanglement based on Topology)
    # If adj[i,j] == 1, we apply an entangling gate between qubit i and j
    for i in range(num_nodes):
        for j in range(i + 1, num_nodes):
            if adj[i, j] == 1:
                qml.CZ(wires=[i, j])

    # Phase 3: Parameterized Convolution Filters (Trainable Rotations)
    for i in range(num_nodes):
        qml.RY(weights[i, 0], wires=i)
        qml.RZ(weights[i, 1], wires=i)

    # Measure the expectation values of each node (predicted future state trend)
    return [qml.expval(qml.PauliZ(i)) for i in range(num_nodes)]

###Step 3: Predictive Training Loop
We train the QGNN using a sliding window approach. The model takes the signal at time $t$ and attempts to predict the clean signal values at time $t+1$. Since it trains predominantly on normal data, it will fail to accurately predict the anomalous states, resulting in a measurable spike in reconstruction error.

In [ ]:
# Initialize weights: 2 parameters per node
weights = np.random.uniform(0, 2 * np.pi, (num_nodes, 2), requires_grad=True)
optimizer = qml.AdamOptimizer(stepsize=0.02)

# Training on normal data profiles (first 40 steps)
print("Training QGNN on normal structural dynamics...")
for epoch in range(31):
    loss = 0
    for t in range(40):
        input_features = data[t]
        target_output = data[t+1]

        # Mean Squared Error Loss Function
        def cost_fn(w):
            predictions = qgnn_layer(input_features, w, adjacency_matrix)
            return np.mean((np.array(predictions) - target_output) ** 2)

        weights = optimizer.step(cost_fn, weights)
        loss += cost_fn(weights)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:2d} | Avg MSE Loss: {loss/40:.5f}")

###Step 4: Evaluating the Anomaly Scores
Now we pass the entire time series (including the window with the injected anomaly) through our trained QGNN to generate an Anomaly Score Profile.

In [ ]:
anomaly_scores = []

for t in range(time_steps - 1):
    preds = qgnn_layer(data[t], weights, adjacency_matrix)
    # Anomaly score is the Mean Absolute Error between prediction and actual next state
    mae = np.mean(np.abs(np.array(preds) - data[t+1]))
    anomaly_scores.append(mae)

print("\n--- Evaluation Results ---")
print(f"Average baseline anomaly score (Normal): {np.mean(anomaly_scores[:40]):.4f}")
print(f"Peak anomaly score (During Injection):  {max(anomaly_scores[48:55]):.4f}")

# Thresholding mechanism
threshold = np.mean(anomaly_scores[:40]) + 3 * np.std(anomaly_scores[:40])
detected_anomalies = [i for i, score in enumerate(anomaly_scores) if score > threshold]
print(f"System flagged structural anomalies at time steps: {detected_anomalies}")

**Topological Expressibility:** How does changing the underlying graph type (e.g., Erdős–Rényi random graphs vs. Scale-free Barabási–Albert graphs) impact the expressibility of the QGNN ansatz?

**Quantum vs. Classical Boundary:** Replace the qgnn_layer with a standard PyTorch Geometric classical GCN layer with equivalent parameters. Benchmark their performance under highly restricted, small-sample training limits. Quantum models often show advantages when training data is extremely scarce.

**Barren Plateaus in Graph Architectures:** As the graph scales up in size ($N > 10$ nodes/qubits), does the variance of the gradients vanish exponentially? Researching how the adjacency matrix topology influences the onset of barren plateaus is an active, open area of QML theory.